# 17.9 近端策略优化 / Proximal Policy Optimization (PPO)

**中文**：REINFORCE 和 Actor-Critic 都有个共同软肋——**数据只能用一次**(on-policy),而且**更新步子不好控**:策略一步迈太大就可能"跳崖"崩掉。**PPO(OpenAI, 2017)** 用一个极其巧妙又简单的技巧解决了这两点,成为**当今最流行、最默认的 RL 算法**——从机器人、游戏 AI(OpenAI Five 打 Dota)到**用 RLHF 对齐 ChatGPT/Claude**,背后都是它。本节从零实现 PPO。
**English**: REINFORCE and Actor-Critic share a weakness — **data can be used only once** (on-policy) — and their **update step is hard to control**: one too-large policy step can "fall off a cliff" and collapse. **PPO (OpenAI, 2017)** fixes both with a strikingly clever yet simple trick, becoming the **most popular, default RL algorithm today** — from robotics and game AI (OpenAI Five at Dota) to **aligning ChatGPT/Claude via RLHF**, PPO is behind it. We implement PPO from scratch.

---

**中文**：PPO 想安全地**用同一批数据更新策略很多次**(提高样本效率)。但反复更新会让新策略 $\pi_\theta$ 越走越远离采样时的旧策略 $\pi_{\text{old}}$,一旦偏太多,用旧数据估的梯度就不可信了、甚至把策略推崩。PPO 的核心是**裁剪的替代目标(clipped surrogate objective)**——**不让策略一次更新走得太远**:
**English**: PPO wants to safely **update the policy many times on the same batch** (better sample efficiency). But repeated updates drift the new policy $\pi_\theta$ far from the old sampling policy $\pi_{\text{old}}$; drift too far and gradients estimated from stale data become untrustworthy, even collapsing the policy. PPO's core is the **clipped surrogate objective** — **don't let the policy move too far in one update**:

$$L^{\text{CLIP}}(\theta)=\mathbb E\Big[\min\big(r_t(\theta)\,A_t,\ \ \text{clip}(r_t(\theta),1-\epsilon,1+\epsilon)\,A_t\big)\Big],\qquad r_t(\theta)=\frac{\pi_\theta(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)}$$

**中文**：逐项拆解:
**English**: Breaking it down:
- **概率比 $r_t$**:新旧策略在该动作上的概率之比。$r_t>1$ 表示新策略更爱这个动作。
  **Probability ratio $r_t$**: ratio of new to old policy on that action. $r_t>1$ means the new policy favors it more.
- **$r_t\cdot A_t$**:标准策略梯度目标(优势大就多提概率)。
  **$r_t\cdot A_t$**: the standard policy-gradient objective (raise probability more when advantage is larger).
- **裁剪 $\text{clip}(r_t,1-\epsilon,1+\epsilon)$**:把比值**限制在 $[1-\epsilon,1+\epsilon]$**(常 $\epsilon=0.2$)。取 $\min$ 意味着:当更新已经让比值超出这个区间(策略走太远),梯度就被**削平、不再给激励**——相当于给策略更新装了**"安全带"**。
  **Clipping $\text{clip}(r_t,1-\epsilon,1+\epsilon)$**: constrains the ratio to $[1-\epsilon,1+\epsilon]$ (often $\epsilon=0.2$). Taking the $\min$ means: once an update pushes the ratio outside this band (policy moved too far), the gradient is **flattened — no more incentive** — a **"seatbelt"** on policy updates.

**中文**：有了这条安全带,PPO 就敢**对同一批数据跑很多轮(epoch)梯度更新**——数据被反复利用,样本效率大增,又不会因走太远而崩。配合 **GAE(17.8 的优势估计)**,就是完整的 PPO。
**English**: With this seatbelt, PPO dares to **run many epochs of gradient updates on the same batch** — reusing data for much better sample efficiency, without collapsing from over-large steps. Combined with **GAE (the advantage estimation from 17.8)**, that is full PPO.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 最火, RLHF 必考）**
> **中文**：**PPO=裁剪替代目标 + 同批数据多轮更新 + GAE**。**裁剪**把新旧策略概率比限制在 $[1{-}\epsilon,1{+}\epsilon]$(min 取悲观值), 相当于**软信任域**——不让策略一步走太远(比 TRPO 的二阶约束简单太多)。**为什么能多轮复用数据**:裁剪保证即使多次更新, 策略也不会漂离采样分布太远。损失=裁剪策略损失 + 价值损失 + 熵正则。**PPO 主导原因**:简单、稳、够用、超参不敏感。**仍是 on-policy**(每轮要重新采样)。**RLHF**:把奖励模型的打分当 reward, 用 PPO 优化语言模型策略——ChatGPT/Claude 的对齐核心。
> **English**: **PPO = clipped surrogate objective + multiple epochs on the same batch + GAE**. **Clipping** constrains the new/old probability ratio to $[1{-}\epsilon,1{+}\epsilon]$ (min takes the pessimistic value), a **soft trust region** — don't let the policy move too far in one step (far simpler than TRPO's second-order constraint). **Why data can be reused for multiple epochs**: clipping keeps the policy from drifting too far from the sampling distribution even across updates. Loss = clipped policy loss + value loss + entropy bonus. **Why PPO dominates**: simple, stable, good enough, hyperparameter-robust. **Still on-policy** (resample each iteration). **RLHF**: treat a reward model's score as the reward and optimize the language-model policy with PPO — the core of ChatGPT/Claude alignment.


In [ ]:

# ============================================================
# 环境 + Actor-Critic 网络 / CartPole + Actor-Critic net
# ============================================================
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, random, time, matplotlib.pyplot as plt
def set_seed(x): torch.manual_seed(x); np.random.seed(x); random.seed(x)
class CartPole:
    g=9.8; mc=1.0; mp=0.1; l=0.5; fm=10.0; tau=0.02
    def reset(s): s.state=np.random.uniform(-0.05,0.05,4); s.steps=0; return s.state.copy()
    def step(s,a):
        x,xd,th,thd=s.state; force=s.fm if a==1 else -s.fm
        ct,st=np.cos(th),np.sin(th); tot=s.mc+s.mp
        temp=(force+s.mp*s.l*thd**2*st)/tot
        thacc=(s.g*st-ct*temp)/(s.l*(4/3-s.mp*ct**2/tot)); xacc=temp-s.mp*s.l*thacc*ct/tot
        x+=s.tau*xd; xd+=s.tau*xacc; th+=s.tau*thd; thd+=s.tau*thacc
        s.state=np.array([x,xd,th,thd]); s.steps+=1
        done=abs(x)>2.4 or abs(th)>12*np.pi/180 or s.steps>=500
        return s.state.copy(),1.0,done
class AC(nn.Module):
    def __init__(s): super().__init__(); s.t=nn.Sequential(nn.Linear(4,128),nn.ReLU()); s.pi=nn.Linear(128,2); s.v=nn.Linear(128,1)
    def forward(s,x): h=s.t(x); return F.softmax(s.pi(h),-1), s.v(h).squeeze(-1)

def gae(rews, vals, dones, last_v, gamma=0.99, lam=0.95):    # 广义优势估计 / GAE
    adv=np.zeros(len(rews)); g=0.0
    for t in reversed(range(len(rews))):
        nv = last_v if t==len(rews)-1 else vals[t+1]
        delta = rews[t] + gamma*nv*(1-dones[t]) - vals[t]     # TD 误差 / TD residual
        g = delta + gamma*lam*(1-dones[t])*g                  # λ 加权累积 / λ-weighted accumulation
        adv[t]=g
    return adv
print("PPO 组件就绪 / ready")


**中文**：从零实现 PPO。流程:①用**旧策略**采集一大批(horizon 步)转移，记下当时的 $\log\pi_{\text{old}}$;②用 GAE 算优势;③对这批数据跑 **K 轮** minibatch 更新，每次用裁剪目标。把"是否裁剪"做成开关以便消融。
**English**: Implement PPO from scratch. Flow: ① collect a large batch (horizon steps) with the **old policy**, recording $\log\pi_{\text{old}}$; ② compute advantages via GAE; ③ run **K epochs** of minibatch updates on that batch using the clipped objective. Make clipping a toggle for the ablation.


In [ ]:

# ============================================================
# 从零实现 PPO / PPO from scratch (with clip on/off switch)
# ============================================================
def collect(net, env, horizon=2048):
    S,Ac,R,D,LP,V=[],[],[],[],[],[]; s=env.reset(); ep_ret=[]; cur=0
    for _ in range(horizon):
        st=torch.tensor(s,dtype=torch.float32)
        with torch.no_grad():
            probs,v=net(st); dist=torch.distributions.Categorical(probs); a=dist.sample()
        S.append(s); Ac.append(int(a)); LP.append(float(dist.log_prob(a))); V.append(float(v))
        s,r,done=env.step(int(a)); R.append(r); D.append(float(done)); cur+=r
        if done: s=env.reset(); ep_ret.append(cur); cur=0
    with torch.no_grad(): last_v=float(net(torch.tensor(s,dtype=torch.float32))[1])
    return (np.array(S,dtype=np.float32),np.array(Ac),np.array(R,dtype=np.float32),
            np.array(D,dtype=np.float32),np.array(LP,dtype=np.float32),np.array(V,dtype=np.float32),last_v,ep_ret)

def train_ppo(iters=50, epochs=20, clip=0.2, use_clip=True, seed=0):
    set_seed(seed); env=CartPole(); net=AC(); opt=torch.optim.Adam(net.parameters(), lr=3e-3); hist=[]
    for it in range(iters):
        S,Ac,R,D,LP,V,last_v,ep_ret = collect(net,env)         # 用旧策略采一批 / rollout with old policy
        adv=gae(R,V,D,last_v); ret=adv+V                        # GAE 优势 + 回报目标 / advantages + returns
        adv=(adv-adv.mean())/(adv.std()+1e-8)                   # 标准化优势 / normalize
        St,At=torch.tensor(S),torch.tensor(Ac); LPt=torch.tensor(LP)
        Advt=torch.tensor(adv,dtype=torch.float32); Rett=torch.tensor(ret,dtype=torch.float32)
        n=len(S)
        for _ in range(epochs):                                 # 同一批数据多轮更新 / K epochs of reuse
            idx=torch.randperm(n)
            for b in range(0,n,64):
                mb=idx[b:b+64]
                probs,v=net(St[mb]); dist=torch.distributions.Categorical(probs); lp=dist.log_prob(At[mb])
                ratio=torch.exp(lp - LPt[mb])                   # 概率比 r_t = π_new/π_old / prob ratio
                if use_clip:
                    unclipped=ratio*Advt[mb]
                    clipped=torch.clamp(ratio,1-clip,1+clip)*Advt[mb]
                    policy_loss=-torch.min(unclipped,clipped).mean()   # 裁剪替代目标(取悲观min)/ clipped surrogate
                else:
                    policy_loss=-(ratio*Advt[mb]).mean()               # 不裁剪(危险)/ unclipped (dangerous)
                value_loss=F.mse_loss(v, Rett[mb]); ent=dist.entropy().mean()
                loss=policy_loss + 0.5*value_loss - 0.01*ent
                opt.zero_grad(); loss.backward(); opt.step()
        hist.append(np.mean(ep_ret) if ep_ret else (hist[-1] if hist else 0))
    return hist

t=time.time(); ppo_hist=train_ppo(50, epochs=20, use_clip=True)
print(f"PPO(裁剪): 最后10轮平均回合回报 {np.mean(ppo_hist[-10:]):.0f} (满分 500) | {time.time()-t:.0f}s")
print("→ PPO 稳稳解决 CartPole, 且靠数据复用样本效率高 / PPO solves CartPole, reusing data efficiently")


**中文**：现在做**核心消融**——**关掉裁剪**。同样"对一批数据跑 20 轮更新",看没有那条"安全带"会发生什么。这直接检验裁剪到底有多重要。
**English**: Now the **core ablation** — **turn off clipping**. With the same "20 epochs of updates per batch," see what happens without the "seatbelt." A direct test of how crucial clipping is.


In [ ]:

# ============================================================
# 消融:去掉裁剪 / ablation: no clipping
# ============================================================
t=time.time(); noclip_hist=train_ppo(50, epochs=20, use_clip=False)
print(f"{'配置/config':<20}{'最后10轮平均回报':>16}{'峰值后最大回撤':>16}")
print(f"{'PPO 裁剪 clip':<20}{np.mean(ppo_hist[-10:]):>16.0f}{max(ppo_hist)-ppo_hist[-1]:>16.0f}")
print(f"{'无裁剪 no-clip':<20}{np.mean(noclip_hist[-10:]):>16.0f}{max(noclip_hist)-noclip_hist[-1]:>16.0f}")
print(f"用时 {time.time()-t:.0f}s")
print("诚实观察:无裁剪时策略会'冲过头'——先学起来又突然崩溃(大回撤)/ no-clip peaks then collapses")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(14,4.7))
# ① 裁剪 vs 无裁剪 学习曲线 / clip vs no-clip
ax[0].plot(ppo_hist,"o-",color="#4C72B0",ms=3,label=f"PPO 裁剪 clip ({np.mean(ppo_hist[-10:]):.0f})")
ax[0].plot(noclip_hist,"o-",color="#C44E52",ms=3,label=f"无裁剪 no-clip ({np.mean(noclip_hist[-10:]):.0f})")
ax[0].axhline(500,ls=":",color="gray")
ax[0].set_title("裁剪=安全带:防止策略冲崩 / clip prevents collapse"); ax[0].set_xlabel("PPO 迭代 iteration"); ax[0].set_ylabel("回合回报 episode return"); ax[0].legend(fontsize=9)
# ② 裁剪机制示意 / the clipping mechanism
r=np.linspace(0,2,200); eps=0.2
for A,c,lab in [(1.0,"#55A868","A>0 (好动作 good)"),(-1.0,"#C44E52","A<0 (坏动作 bad)")]:
    obj=np.minimum(r*A, np.clip(r,1-eps,1+eps)*A)
    ax[1].plot(r,obj,color=c,lw=2,label=lab)
ax[1].axvspan(1-eps,1+eps,alpha=0.1,color="gray")
ax[1].axvline(1,ls=":",color="k")
ax[1].set_title("裁剪目标 L^CLIP(超出[0.8,1.2]梯度被削平)"); ax[1].set_xlabel("概率比 r=π_new/π_old"); ax[1].set_ylabel("目标值 objective"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/rl09_viz.png",dpi=80); plt.show()
print("右图:好动作(绿)推概率到1.2就封顶、坏动作(红)压到0.8就封顶——策略被限制在旧策略附近")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **裁剪是 PPO 的灵魂**:两者都对同一批数据跑 20 轮更新,但**带裁剪的稳稳到满分 500 且不回撤;不裁剪的先学起来、随后突然崩溃(大幅回撤)**。原因正如设计:多轮复用同一批数据时,不裁剪的策略会**越更新越偏离采样时的旧策略**,旧数据估的梯度失真,把策略推过头就崩。裁剪像安全带,把每步策略变化锁在旧策略附近,于是敢放心复用数据。
2. **右图看懂裁剪机制**:对好动作($A>0$),概率比涨到 $1+\epsilon=1.2$ 就**封顶**(不再给继续提高的激励);对坏动作($A<0$),压到 $1-\epsilon=0.8$ 就封顶。取 $\min$(悲观)保证它**只在"没走太远"时给完整梯度**。这就是"软信任域"——比 TRPO 的二阶 KL 约束简单太多,却同样有效。
3. **PPO 为什么统治 RL**:①**简单**(几十行核心代码,一阶优化);②**稳**(裁剪防崩、超参不敏感);③**样本效率还行**(数据复用多轮)。所以从机器人、游戏到 **RLHF 对齐大模型**都用它。诚实局限:仍是 **on-policy**(每轮要重采样,不如 off-policy 的 SAC/DQN 省数据);CartPole 太简单,PPO 的真正威力在高维连续控制(MuJoCo、LunarLander)和 LLM 对齐上。

**English**:
1. **Clipping is PPO's soul**: both run 20 epochs on the same batch, but **with clipping it steadily hits the max 500 with no drawdown; without clipping it learns then suddenly collapses (a big drawdown)**. Exactly as designed: reusing one batch for many epochs, the unclipped policy **drifts further from the old sampling policy each update**, gradients from stale data distort, and overshooting collapses the policy. Clipping is a seatbelt locking each step near the old policy, so we can safely reuse data.
2. **The right plot explains the mechanism**: for good actions ($A>0$), the ratio's benefit **caps at $1+\epsilon=1.2$** (no more incentive to raise further); for bad actions ($A<0$), it caps at $1-\epsilon=0.8$. Taking the $\min$ (pessimistic) ensures **full gradient only while "not too far."** This is the "soft trust region" — far simpler than TRPO's second-order KL constraint, yet as effective.
3. **Why PPO rules RL**: ① **simple** (tens of lines, first-order optimization); ② **stable** (clip prevents collapse, hyperparameter-robust); ③ **decent sample efficiency** (multi-epoch data reuse). So it powers everything from robotics and games to **RLHF alignment of LLMs**. Honest limits: still **on-policy** (resample each iteration, less data-thrifty than off-policy SAC/DQN); CartPole is too easy — PPO's real power shows in high-dim continuous control (MuJoCo, LunarLander) and LLM alignment.

> 💼 **实战视角 / Practical angle**
> **中文**:PPO 是**工业与研究的默认 RL 算法**。落地关键点:①**GAE 的 λ≈0.95、裁剪 ε≈0.2、每批 3~10 epoch** 是常用配方;②**优势标准化**、**价值损失裁剪**、**梯度裁剪**、**熵正则**都是稳定训练的常见 trick;③**RLHF**:reward=奖励模型打分,还要加一项 **KL 惩罚**(别让语言模型偏离原始模型太远)——本质和裁剪一个思想(信任域)。④连续动作:策略输出高斯的均值和方差。面试金句:*"PPO 用裁剪把新旧策略比锁在 [1-ε,1+ε], 实现软信任域, 从而能对同批数据多轮更新(样本效率)又不崩; 简单稳定是它统治 RL(含 RLHF)的原因。"*
> **English**: PPO is the **default RL algorithm in industry and research**. Deployment keys: ① common recipe: **GAE λ≈0.95, clip ε≈0.2, 3–10 epochs per batch**; ② **advantage normalization, value-loss clipping, gradient clipping, entropy bonus** are standard stabilizers; ③ **RLHF**: reward = reward-model score, plus a **KL penalty** (keep the LM from drifting too far from the base model) — the same trust-region idea as clipping; ④ continuous actions: the policy outputs a Gaussian mean and variance. Interview line: *"PPO clips the new/old policy ratio to [1-ε,1+ε] for a soft trust region, so it can reuse a batch for multiple epochs (sample efficiency) without collapsing; simplicity and stability are why it dominates RL, including RLHF."*

---
### 小结 / Summary
- **中文**:PPO=裁剪替代目标 + 同批多轮更新 + GAE; 裁剪把新旧策略比锁在 [1-ε,1+ε] 实现软信任域。
- **English**: PPO = clipped surrogate objective + multi-epoch reuse + GAE; clipping locks the new/old ratio to [1-ε,1+ε] as a soft trust region.
- **中文**:消融证明裁剪是灵魂——无裁剪多轮复用会冲过头崩溃, 有裁剪稳到满分。
- **English**: The ablation proves clipping is the soul — without it, multi-epoch reuse overshoots and collapses; with it, steadily maxes out.
- **中文**:简单+稳定+够用→PPO 成为默认 RL 算法, 也是 RLHF 对齐 ChatGPT/Claude 的核心。
- **English**: Simple + stable + good enough → PPO is the default RL algorithm and the core of RLHF alignment for ChatGPT/Claude.
